# Meeting Transcript Summarizer

This summarizer splits the transcript into sentences, turns each sentence into a tf-idf vector that is then used to compute the cosine similarity between each sentence pair. Then each sentence is scored by how connected it is to the rest of the meeting, and finally the top-k sentences are returned in their original order.

We use ROUGE-style evaluation to compare summaries to hand written ones.

In [1]:
import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## 1. Sentence splitting

In [2]:
SPEAKER = re.compile(r"^\s*[A-Z][\w .'-]{0,30}:\s*")   # e.g. "Alice: " at the start of a line

def split_sentences(transcript, min_words=4):
    """Split a transcript into sentences, removing speaker labels and very short fragments."""
    sentences = []
    for line in transcript.splitlines():
        line = SPEAKER.sub("", line).strip()
        if not line:
            continue
        for s in re.split(r"(?<=[.!?])\s+", line):
            s = s.strip()
            if len(s.split()) >= min_words:   # drop "Yeah." / "Okay, thanks."
                sentences.append(s)
    return sentences

## 2. The summarizer

Score of sentence *i* = average cosine similarity to every *other* sentence. A sentence that shares key vocabulary with much of the meeting is central to it. This is a simplified version of the idea behind graph-based methods like LexRank.

In [3]:
def score_sentences(sentences):
    """Return one centrality score per sentence."""
    n = len(sentences)
    if n < 2:
        return np.ones(n)
    vec = TfidfVectorizer(stop_words="english", sublinear_tf=True)
    try:
        X = vec.fit_transform(sentences)          # (n_sentences, n_terms), L2-normalised rows
    except ValueError:                            # every word was a stop word
        return np.zeros(n)
    sim = cosine_similarity(X)                    # (n, n) pairwise similarity
    np.fill_diagonal(sim, 0.0)                    # ignore self-similarity
    return sim.sum(axis=1) / (n - 1)              # mean similarity to the rest


def summarize(transcript, k=3, min_words=4):
    """Return the k most central sentences, in original order."""
    sentences = split_sentences(transcript, min_words)
    if len(sentences) <= k:
        return sentences
    scores = score_sentences(sentences)
    top = np.argsort(scores)[::-1][:k]            # indices of the k best scores
    return [sentences[i] for i in sorted(top)]    # restore meeting order

## 3. Trying it on a sample meeting

In [4]:
MEETING_1 = """
Alice: Okay, let's get started with the product launch review.
Bob: The launch date is currently set for November 15, and engineering says the launch is on track.
Alice: Great. Marketing needs the final launch copy by October 25 so the campaign can start on time.
Carol: I can deliver the launch copy by October 25, but I need the final pricing first.
Bob: Pricing is still open. The pricing decision depends on the cost of cloud hosting.
Dave: Cloud hosting costs came in 10 percent higher than expected, so we may need to raise the pricing.
Alice: Then let's decide on pricing this Thursday so Carol can finish the launch copy.
Carol: Sounds good. Also, the office coffee machine is broken again.
Bob: Yeah. Anyway, QA found two blocking bugs in the checkout flow that must be fixed before the November 15 launch.
Dave: I will fix the checkout bugs by next Wednesday.
Alice: Perfect. To recap: decide pricing Thursday, copy by October 25, checkout bugs fixed by Wednesday.
"""

for s in summarize(MEETING_1, k=3):
    print("-", s)

- I can deliver the launch copy by October 25, but I need the final pricing first.
- Then let's decide on pricing this Thursday so Carol can finish the launch copy.
- To recap: decide pricing Thursday, copy by October 25, checkout bugs fixed by Wednesday.


## 4. Evaluation harness

We compare a candidate summary to a hand-written reference with **ROUGE-1** (unigram overlap) and **ROUGE-2** (bigram overlap), implemented from scratch with `numpy`-free counting. We also report two baselines, **lead-k** (first *k* sentences) and **random-k**, so the numbers have context.

In [5]:
from collections import Counter

def tokens(text):
    return re.findall(r"[a-z0-9']+", text.lower())

def ngrams(toks, n):
    return Counter(tuple(toks[i:i + n]) for i in range(len(toks) - n + 1))

def rouge_n(candidate, reference, n=1):
    """Return (precision, recall, f1) of n-gram overlap."""
    c, r = ngrams(tokens(candidate), n), ngrams(tokens(reference), n)
    overlap = sum((c & r).values())
    p = overlap / max(sum(c.values()), 1)
    rec = overlap / max(sum(r.values()), 1)
    f1 = 2 * p * rec / (p + rec) if p + rec else 0.0
    return p, rec, f1

def lead_k(transcript, k=3):
    return split_sentences(transcript)[:k]

def random_k(transcript, k=3, seed=0):
    sents = split_sentences(transcript)
    rng = np.random.default_rng(seed)
    idx = sorted(rng.choice(len(sents), size=min(k, len(sents)), replace=False))
    return [sents[i] for i in idx]

def evaluate(summarizers, dataset, k=3):
    """summarizers: {name: fn(transcript, k) -> list[str]}; dataset: list of {transcript, reference}."""
    results = {}
    for name, fn in summarizers.items():
        r1, r2 = [], []
        for ex in dataset:
            summary = " ".join(fn(ex["transcript"], k))
            r1.append(rouge_n(summary, ex["reference"], 1)[2])
            r2.append(rouge_n(summary, ex["reference"], 2)[2])
        results[name] = (float(np.mean(r1)), float(np.mean(r2)))
    return results

def print_report(results):
    print(f"{'method':<12}{'ROUGE-1 F1':>12}{'ROUGE-2 F1':>12}")
    for name, (a, b) in sorted(results.items(), key=lambda kv: -kv[1][0]):
        print(f"{name:<12}{a:>12.3f}{b:>12.3f}")

### Hand-written references

Other meeting examples

In [6]:
MEETING_2 = """
Erin: Today we're reviewing the hiring plan for the data team.
Frank: We have two open data engineer roles and one open analyst role.
Erin: The data engineer roles are the priority because the pipeline migration is blocked without them.
Grace: Recruiting has 12 candidates for the data engineer roles, and interviews start next Monday.
Frank: I suggest a take-home exercise for data engineer candidates to speed up the technical screen.
Erin: Agreed, but keep the exercise under two hours so we do not lose good candidates.
Grace: The analyst role can wait until January because the budget for it is not approved yet.
Hank: By the way, is the team lunch still on Friday?
Erin: Yes, Friday. Back to hiring: Grace will schedule the data engineer interviews and Frank will write the take-home exercise.
"""

DATASET = [
    {"transcript": MEETING_1,
     "reference": "The November 15 launch is on track but pricing is undecided because cloud hosting costs rose. "
                  "Pricing will be decided Thursday so Carol can deliver launch copy by October 25, "
                  "and Dave will fix the blocking checkout bugs by Wednesday."},
    {"transcript": MEETING_2,
     "reference": "The team will prioritize hiring two data engineers because the pipeline migration is blocked. "
                  "Interviews start Monday with a short take-home exercise, "
                  "while the analyst role waits until January for budget approval."},
]

In [7]:
SUMMARIZERS = {
    "tfidf":  summarize,
    "lead":   lead_k,
    "random": random_k,
}

results = evaluate(SUMMARIZERS, DATASET, k=3)
print_report(results)

method        ROUGE-1 F1  ROUGE-2 F1
tfidf              0.458       0.175
lead               0.422       0.188
random             0.350       0.119


### Inspect a second meeting

In [8]:
for s in summarize(MEETING_2, k=3):
    print("-", s)

- We have two open data engineer roles and one open analyst role.
- Recruiting has 12 candidates for the data engineer roles, and interviews start next Monday.
- Back to hiring: Grace will schedule the data engineer interviews and Frank will write the take-home exercise.


## Where to go next
- **Redundancy control:** use MMR (pick sentences that are central *and* dissimilar to those already chosen).
- **PageRank over the similarity matrix** instead of the plain mean (this is LexRank).
- **Tune `k`** as a fraction of meeting length instead of a fixed number.
- **More references:** the two examples above are far too few to draw conclusions; ROUGE only becomes meaningful with a larger set.